# Automatic false-positive cleanup

Run this **after** `run_in_colab.ipynb` has produced `outputs/detections/*.csv`.

Three automatic filters — no manual listening required:

1. **Mahalanobis OOD** — distance of each detection's model feature vector to
   the mean of its predicted class in the training set. A real *Pan* call lives
   close to the training *Pan* cluster; a bird call does not.
2. **YAMNet cross-check** — Google's 521-class audio tagger is run on every
   detection clip. If its top classes are `Bird`, `Chirp, tweet`, `Insect`,
   `Wind`, `Rain`, `Speech`, etc. the window is flagged.
3. **Temporal isolation** — primates usually call in bouts. A detection with
   zero same-species neighbours within +/-30 s on the same recording is
   suspicious.

A detection is kept only if **all three** filters agree it's real. Detections
flagged by **>=2** filters are auto-copied into `auto_flagged_fp/<reason>/` so
the next retraining iteration can fold them into the background class as hard
negatives.

## Step 0 — GPU check

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs:       {gpus}')
if not gpus:
    print('\nWARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')

## Step 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone the repo (claude/review-repo-QAc6j)

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q -b claude/review-repo-QAc6j https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!git log --oneline -3

## Step 3 — Install deps (adds tensorflow-hub for YAMNet)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q tensorflow-hub resampy
print('\nDone.')

## Step 4 — Import project modules

In [ ]:
import os, sys, importlib
PROJECT_ROOT = '/content/primates-sound-detection'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ['PRIMATE_DATA_ROOT'] = '/content/drive/MyDrive/primates-data'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
from pathlib import Path
from collections import defaultdict

import config, data_loader, preprocessing, model as model_module, detection
for m in (config, data_loader, preprocessing, model_module, detection):
    importlib.reload(m)

CLEANUP_DIR = Path(config.OUTPUT_ROOT) / 'auto_cleanup'
CLEANUP_DIR.mkdir(parents=True, exist_ok=True)
FP_DIR = CLEANUP_DIR / 'auto_flagged_fp'
FP_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cleanup output: {CLEANUP_DIR}')

## Step 5 — Load the trained model + build a feature extractor

We tap the `dense_256` layer of the VGG19 head to get a 256-dim feature vector per window. This is what Mahalanobis OOD will operate on.

In [ ]:
best_model_path = os.path.join(config.MODEL_SAVE_DIR, 'best_model.h5')
assert os.path.exists(best_model_path), f'No trained model at {best_model_path}. Run run_in_colab.ipynb first.'

model = model_module.load_trained_model(best_model_path)

# Build a feature extractor that taps the dense_256 layer.
# Keras 3 removed Layer.output_shape; we rebuild the sub-model via get_layer
# + model.input (which still works for Functional models) and report the
# shape from the new Model wrapper instead of the layer.
feat_layer = model.get_layer('dense_256')
feature_extractor = tf.keras.Model(inputs=model.inputs, outputs=feat_layer.output)
print(f'Feature extractor: input={feature_extractor.input_shape}, '
      f'output={feature_extractor.output_shape}')


## Step 6 — Load training data once

Used both to compute the Mahalanobis class stats and to verify the training-set feature distribution is tight.

In [ ]:
species_data = data_loader.load_species_data()
background_data = data_loader.load_background_data()
data_loader.print_data_summary(species_data, background_data)

## Step 7 — Compute per-class Mahalanobis statistics

For each class *c*, we compute the mean `mu_c` and a shared (pooled) inverse
covariance `Sigma^{-1}` over the 256-dim features. Lee et al. (NeurIPS 2018)
show that a pooled covariance works better than per-class covariance when the
training set is small (which is our situation).

Distance of a test feature `f` to class `c`:

    d^2(f, c) = (f - mu_c)^T Sigma^{-1} (f - mu_c)

We only care about the distance to the **predicted** class — it measures
"does this window look like a typical member of the class the model assigned
it to, or is it an outlier?" 

In [ ]:
def audio_to_model_input(audio, sr):
    img = preprocessing.preprocess_audio(audio, sr)
    return preprocessing.preprocess_for_model(img)

STATS_PATH = CLEANUP_DIR / 'class_stats.npz'
if STATS_PATH.exists():
    print(f'Loading cached stats from {STATS_PATH}')
    z = np.load(STATS_PATH)
    class_means = z['class_means']
    inv_cov = z['inv_cov']
    train_feats_by_class = {int(k): z[f'feats_{int(k)}'] for k in z['class_ids']}
else:
    print('Extracting features for all training clips...')
    # Build (X, y) using only the *original* (non-augmented) clips — features
    # for augmented copies are nearly identical and would just waste time.
    label_map = {name: i for i, name in enumerate(config.CLASS_NAMES)}
    X_list, y_list = [], []
    for sp_name, audio_list in species_data.items():
        for audio, _ in audio_list:
            X_list.append(audio_to_model_input(audio, config.SAMPLE_RATE))
            y_list.append(label_map[sp_name])
    for audio, _ in background_data:
        X_list.append(audio_to_model_input(audio, config.SAMPLE_RATE))
        y_list.append(label_map['Background'])
    X_arr = np.array(X_list, dtype=np.float32)
    y_arr = np.array(y_list, dtype=np.int64)
    print(f'  training tensor: {X_arr.shape}')

    feats = feature_extractor.predict(X_arr, batch_size=config.BATCH_SIZE, verbose=1)
    print(f'  feature tensor:  {feats.shape}')

    class_ids = sorted(np.unique(y_arr).tolist())
    class_means = np.zeros((len(config.CLASS_NAMES), feats.shape[1]), dtype=np.float32)
    train_feats_by_class = {}
    centered_parts = []
    for c in class_ids:
        fc = feats[y_arr == c]
        mu = fc.mean(axis=0)
        class_means[c] = mu
        train_feats_by_class[c] = fc
        centered_parts.append(fc - mu)
    centered = np.concatenate(centered_parts, axis=0)
    # Pooled covariance + ridge for numerical stability
    cov = np.cov(centered, rowvar=False) + 1e-4 * np.eye(centered.shape[1], dtype=np.float32)
    inv_cov = np.linalg.inv(cov).astype(np.float32)

    save_kwargs = {'class_means': class_means, 'inv_cov': inv_cov, 'class_ids': np.array(class_ids)}
    for c, fc in train_feats_by_class.items():
        save_kwargs[f'feats_{c}'] = fc
    np.savez(STATS_PATH, **save_kwargs)
    print(f'Saved stats to {STATS_PATH}')

def mahalanobis(features, class_idx):
    diff = features - class_means[class_idx]
    return np.einsum('bi,ij,bj->b', diff, inv_cov, diff)

## Step 8 — Calibrate thresholds from in-distribution training data

For each class we compute the 95th percentile of the Mahalanobis distance of its own training features. Anything above that is OOD for that class.

In [ ]:
PERCENTILE = 95
class_thresholds = {}
fig, axes = plt.subplots(1, len(config.CLASS_NAMES), figsize=(4.5 * len(config.CLASS_NAMES), 3.5))
for c, name in enumerate(config.CLASS_NAMES):
    fc = train_feats_by_class[c]
    d2 = mahalanobis(fc, c)
    thr = float(np.percentile(d2, PERCENTILE))
    class_thresholds[c] = thr
    ax = axes[c]
    ax.hist(d2, bins=40, color='#2980B9', alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axvline(thr, color='#C0392B', lw=2, label=f'p{PERCENTILE}={thr:.1f}')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Mahalanobis d^2')
    ax.legend(fontsize=9)
plt.tight_layout()
out = CLEANUP_DIR / 'mahalanobis_in_distribution.png'
plt.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'\nPer-class thresholds (p{PERCENTILE}):')
for c, name in enumerate(config.CLASS_NAMES):
    print(f'  {name:30s} {class_thresholds[c]:.2f}')

## Step 9 — Load every detection CSV + index the source audio

Each row is one (file, start, end, species, confidence) detection emitted at the threshold chosen in `run_in_colab.ipynb`.

In [ ]:
det_dir = Path(config.DETECTION_OUTPUT_DIR)
csv_files = sorted(det_dir.glob('*_detections.csv'))
assert csv_files, f'No detection CSVs in {det_dir}. Run run_in_colab.ipynb first.'

long_audio_files = {os.path.basename(p): p for p in data_loader.get_long_audio_files()}

all_rows = []
for csv in csv_files:
    df = pd.read_csv(csv)
    if len(df) == 0:
        continue
    source_name = csv.stem.replace('_detections', '') + '.wav'
    df['source_file'] = source_name
    df['source_path'] = long_audio_files.get(source_name, '')
    all_rows.append(df)

if not all_rows:
    raise SystemExit('No detections found in any CSV - nothing to clean up.')

det_df = pd.concat(all_rows, ignore_index=True)
det_df['det_id'] = np.arange(len(det_df))
print(f'{len(det_df)} detections across {det_df["source_file"].nunique()} files')
print(det_df.groupby('species').size().to_string())

## Step 10 — Cut each detection clip into a 5 s numpy window

Cached by `source_file` so each long recording is only loaded once.

In [ ]:
audio_cache = {}
clip_len = int(round(config.WINDOW_SIZE * config.SAMPLE_RATE))

def get_clip(source_path, start_time):
    if not source_path or not os.path.exists(source_path):
        return None
    if source_path not in audio_cache:
        y, sr = librosa.load(source_path, sr=config.SAMPLE_RATE, mono=True)
        audio_cache[source_path] = y
    y = audio_cache[source_path]
    s = int(round(start_time * config.SAMPLE_RATE))
    clip = y[s:s + clip_len]
    if len(clip) < clip_len:
        clip = np.pad(clip, (0, clip_len - len(clip)))
    return clip

clips = []
missing = 0
for row in det_df.itertuples():
    c = get_clip(row.source_path, row.start_time)
    if c is None:
        missing += 1
        c = np.zeros(clip_len, dtype=np.float32)
    clips.append(c)
print(f'Extracted {len(clips)} clips ({missing} missing source audio)')
print(f'Audio cache: {len(audio_cache)} file(s) held in memory')

## Filter 1 — Mahalanobis OOD

For every detection, compute its feature vector, score it against the class the model predicted, and compare to that class's calibrated threshold.

In [ ]:
X_det = np.stack([audio_to_model_input(c, config.SAMPLE_RATE) for c in clips]).astype(np.float32)
det_feats = feature_extractor.predict(X_det, batch_size=config.BATCH_SIZE, verbose=1)

label_map = {name: i for i, name in enumerate(config.CLASS_NAMES)}
pred_class_ids = det_df['species'].map(label_map).to_numpy()

mahal_scores = np.zeros(len(det_df), dtype=np.float32)
mahal_flag = np.zeros(len(det_df), dtype=bool)
for i in range(len(det_df)):
    c = int(pred_class_ids[i])
    d2 = float(mahalanobis(det_feats[i:i+1], c)[0])
    mahal_scores[i] = d2
    mahal_flag[i] = d2 > class_thresholds[c]

det_df['mahalanobis_d2'] = mahal_scores
det_df['flag_mahal'] = mahal_flag
print(f'Mahalanobis flagged {int(mahal_flag.sum())} / {len(det_df)} detections')
print(det_df.groupby('species')['flag_mahal'].mean().apply(lambda p: f'{p*100:.1f}%').to_string())

## Filter 2 — YAMNet cross-check

YAMNet is Google's pretrained audio tagger (521 AudioSet classes). It has
never seen our three primates, but it **does** know Bird, Chirp, Insect, Wind,
Rain, Speech, etc. — exactly the sounds we suspect the model is confusing for
primates.

Decision rule: resample each detection to 16 kHz, run YAMNet, take the top
class by mean score. If that top class is in `SUSPICIOUS_YAMNET`, flag it.

In [ ]:
import tensorflow_hub as hub
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
class_map_path = yamnet.class_map_path().numpy().decode('utf-8')
yamnet_classes = pd.read_csv(class_map_path)['display_name'].tolist()
print(f'YAMNet loaded. {len(yamnet_classes)} classes.')

SUSPICIOUS_YAMNET = {
    'Bird', 'Bird vocalization, bird call, bird song', 'Chirp, tweet',
    'Squawk', 'Pigeon, dove', 'Crow', 'Owl', 'Gull, seagull',
    'Insect', 'Cricket', 'Cicada', 'Mosquito', 'Fly, housefly', 'Bee, wasp, etc.',
    'Wind', 'Wind noise (microphone)', 'Rustling leaves', 'Rain', 'Rain on surface',
    'Thunder', 'Thunderstorm', 'Stream', 'Waterfall',
    'Silence', 'Speech', 'Male speech, man speaking', 'Female speech, woman speaking',
    'Conversation', 'Narration, monologue', 'Static', 'White noise', 'Pink noise',
    'Hum', 'Buzz', 'Mains hum',
}

def yamnet_top(clip_44k):
    clip_16k = librosa.resample(clip_44k.astype(np.float32), orig_sr=config.SAMPLE_RATE, target_sr=16000)
    scores, _, _ = yamnet(clip_16k)
    mean_scores = tf.reduce_mean(scores, axis=0).numpy()
    top_idx = int(np.argmax(mean_scores))
    return yamnet_classes[top_idx], float(mean_scores[top_idx])

yam_top_class = []
yam_top_score = []
yam_flag = np.zeros(len(det_df), dtype=bool)
for i, clip in enumerate(clips):
    name, score = yamnet_top(clip)
    yam_top_class.append(name)
    yam_top_score.append(score)
    yam_flag[i] = name in SUSPICIOUS_YAMNET
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(clips)}')

det_df['yamnet_top'] = yam_top_class
det_df['yamnet_score'] = yam_top_score
det_df['flag_yamnet'] = yam_flag
print(f'\nYAMNet flagged {int(yam_flag.sum())} / {len(det_df)} detections')
print('\nTop-5 YAMNet classes across all detections:')
print(pd.Series(yam_top_class).value_counts().head(10).to_string())

## Filter 3 — Temporal isolation

Forest primates typically call in **bouts** (several calls within seconds to a
minute). A detection that has zero same-species neighbours within +/-30 s on
the same recording is likely a one-off noise event rather than a real bout.

In [ ]:
ISOLATION_WINDOW_S = 30.0

det_df = det_df.sort_values(['source_file', 'species', 'start_time']).reset_index(drop=True)
iso_flag = np.zeros(len(det_df), dtype=bool)
n_neighbours = np.zeros(len(det_df), dtype=int)
for (src, sp), grp in det_df.groupby(['source_file', 'species']):
    idxs = grp.index.to_numpy()
    starts = grp['start_time'].to_numpy()
    for j, i in enumerate(idxs):
        # Count other detections (not itself) within the window
        diffs = np.abs(starts - starts[j])
        diffs[j] = np.inf
        n = int((diffs <= ISOLATION_WINDOW_S).sum())
        n_neighbours[i] = n
        iso_flag[i] = (n == 0)

det_df['n_neighbours_30s'] = n_neighbours
det_df['flag_isolated'] = iso_flag
# restore stable order
det_df = det_df.sort_values('det_id').reset_index(drop=True)
print(f'Temporal-isolation flagged {int(iso_flag.sum())} / {len(det_df)} detections')

## Merge flags and split clean vs suspicious

A detection is **clean** only if *all three* filters are happy. A detection is
**suspicious** if any filter raises it. Detections with >=2 filters agreeing
are treated as strong FPs and saved as hard negatives for the next training
iteration.

In [ ]:
flag_cols = ['flag_mahal', 'flag_yamnet', 'flag_isolated']
det_df['n_flags'] = det_df[flag_cols].sum(axis=1).astype(int)

def reason_str(row):
    parts = []
    if row.flag_mahal:    parts.append('mahal')
    if row.flag_yamnet:   parts.append(f'yamnet:{row.yamnet_top}')
    if row.flag_isolated: parts.append('isolated')
    return '|'.join(parts) if parts else ''
det_df['flag_reason'] = det_df.apply(reason_str, axis=1)

clean_df = det_df[det_df['n_flags'] == 0].copy()
suspicious_df = det_df[det_df['n_flags'] > 0].copy()
strong_fp_df = det_df[det_df['n_flags'] >= 2].copy()

clean_csv = CLEANUP_DIR / 'clean_detections.csv'
susp_csv  = CLEANUP_DIR / 'suspicious_detections.csv'
clean_df.drop(columns=['all_probs'], errors='ignore').to_csv(clean_csv, index=False)
suspicious_df.drop(columns=['all_probs'], errors='ignore').to_csv(susp_csv, index=False)

print(f'Total:       {len(det_df)}')
print(f'Clean:       {len(clean_df)}   -> {clean_csv}')
print(f'Suspicious:  {len(suspicious_df)}   -> {susp_csv}')
print(f'Strong FPs (>=2 flags): {len(strong_fp_df)}')
print('\nPer-species breakdown:')
print(det_df.groupby('species').agg(
    total=('det_id', 'count'),
    clean=('n_flags', lambda x: int((x == 0).sum())),
    suspicious=('n_flags', lambda x: int((x > 0).sum())),
    strong_fp=('n_flags', lambda x: int((x >= 2).sum())),
).to_string())

## Save strong-FP audio clips as hard negatives

Clips with **>=2 filters** agreeing are saved under `auto_flagged_fp/<primary_reason>/` as WAVs. These are the clips you fold into the background class before the next retraining run.

In [ ]:
def primary_reason(row):
    # Priority: Mahalanobis > YAMNet > isolated
    if row.flag_mahal:
        return 'mahal'
    if row.flag_yamnet:
        return 'yamnet'
    return 'isolated'

n_saved = 0
for row in strong_fp_df.itertuples():
    clip = clips[row.det_id]
    reason = primary_reason(row)
    sub = FP_DIR / reason
    sub.mkdir(parents=True, exist_ok=True)
    stem = os.path.splitext(row.source_file)[0]
    fname = f'{row.species}__{stem}__t{int(row.start_time):05d}s__conf{row.confidence:.2f}.wav'
    sf.write(sub / fname, clip, config.SAMPLE_RATE)
    n_saved += 1
print(f'Saved {n_saved} strong-FP clips under {FP_DIR}')
for sub in sorted(FP_DIR.iterdir()):
    if sub.is_dir():
        print(f'  {sub.name}/: {len(list(sub.glob("*.wav")))} clips')

## Summary visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) stacked bar: clean / 1-flag / 2-flags / 3-flags per species
order = [s for s in config.CLASS_NAMES if s in det_df['species'].unique()]
counts = pd.DataFrame({
    'clean':   [int(((det_df.species == s) & (det_df.n_flags == 0)).sum()) for s in order],
    '1 flag':  [int(((det_df.species == s) & (det_df.n_flags == 1)).sum()) for s in order],
    '2 flags': [int(((det_df.species == s) & (det_df.n_flags == 2)).sum()) for s in order],
    '3 flags': [int(((det_df.species == s) & (det_df.n_flags == 3)).sum()) for s in order],
}, index=order)
colors = ['#27AE60', '#F1C40F', '#E67E22', '#C0392B']
counts.plot(kind='bar', stacked=True, color=colors, ax=axes[0], edgecolor='black', linewidth=0.5)
axes[0].set_title('Auto-cleanup verdict per species')
axes[0].set_ylabel('Detections')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(fontsize=9)

# (b) flag overlap Venn-ish bar
venn_rows = {
    'Mahal only':   int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'YAMNet only':  int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'Isolated only':int(((~det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'M+Y':          int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (~det_df.flag_isolated)).sum()),
    'M+I':          int(((det_df.flag_mahal) & (~det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'Y+I':          int(((~det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
    'All 3':        int(((det_df.flag_mahal) & (det_df.flag_yamnet) & (det_df.flag_isolated)).sum()),
}
axes[1].bar(list(venn_rows.keys()), list(venn_rows.values()),
            color=['#F1C40F', '#F1C40F', '#F1C40F', '#E67E22', '#E67E22', '#E67E22', '#C0392B'],
            edgecolor='black', linewidth=0.6)
axes[1].set_title('Filter agreement')
axes[1].set_ylabel('Detections')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
out = CLEANUP_DIR / 'auto_cleanup_summary.png'
plt.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Summary chart saved to {out}')

## Done

You now have three deliverables on Drive under `outputs/auto_cleanup/`:

- `clean_detections.csv` — the subset you can trust without listening
- `suspicious_detections.csv` — with a `flag_reason` column explaining *why*
- `auto_flagged_fp/<reason>/*.wav` — strong FPs to fold into the background
  class for the next retraining iteration

**Next retraining tip:** copy everything under `auto_flagged_fp/` into a new
background folder (e.g. `background/auto_hard_negatives/`), add that folder
to `config.BACKGROUND_FOLDERS`, and re-run `run_in_colab.ipynb` from Step 6.
Every iteration should monotonically shrink the false-positive set.